In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%env OPENAI_API_KEY=""
%env HUGGINGFACE_TOKEN = ""
                   
if not 'dirguard' in locals():
  %cd ..
  dirguard = True

%aimport vivabench
%aimport vivabench.util

from vivabench.util import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: OPENAI_API_KEY=""
env: HUGGINGFACE_TOKEN=""


In [3]:
#chatgpt_decode([{'role':'user', 'content': 'Say hello.'}])
#hfapi_decode([{'role':'user', 'content': 'Say hello.'}], model_name='mistralai/Mixtral-8x7B-Instruct-v0.1')
#hfapi_decode([{'role':'user', 'content': 'Say hello.'}], model_name='meta-llama/Llama-2-70b-chat-hf')

In [4]:
with open('prompts/agent.prompt', 'r') as f:
  agent_prompt_template = f.read()
with open('prompts/simulator.prompt', 'r') as f:
  simulator_prompt_template = f.read()
with open('cases/prototype_1.json', 'r') as f:
  test_case = json.load(f)

In [5]:
# set limit to number of exchanges
limit = 5
max_length = 512
# agent_model = 'mistralai/Mixtral-8x7B-Instruct-v0.1'
# examiner_model = 'mistralai/Mixtral-8x7B-Instruct-v0.1' #'meta-llama/Llama-2-70b-chat-hf' # We should probably use GPT-4-turbo here for testing, with the goal of finetuning Mistral 7B to be a capable examiner.

agent_model = 'gpt-4-1106-preview'
examiner_model = 'gpt-4-1106-preview'
i = 0
response_history = []
chat_history = f"Examiner: {test_case['case_prompt']}"

In [8]:
print(chat_history, end='')

while i < limit:
  agent_prompt = str_to_msgs(agent_prompt_template.format(
      case_prompt = test_case['case_prompt'],
      chat_history = chat_history
  ))

  # Agent
  # agent_response = hfapi_decode(agent_prompt, model_name=agent_model, max_length=max_length)
  agent_response = chatgpt_decode(agent_prompt, model_name=agent_model, max_length=max_length)
    
  response_history.append(agent_response)
  assert 'Response to examiner:' in agent_response, "Agent response must include 'Response to examiner:'"
  agent_response = agent_response.split('Response to examiner:')[-1].strip()

  chat_history += f"\n\nStudent: {agent_response}"
  print(f"\n\nStudent: {agent_response}", end='')

  # Examiner
  simulator_prompt = str_to_msgs(simulator_prompt_template.format(
      case_prompt = test_case['case_prompt'],
      further_information = test_case['further_information'],
      case_answer = test_case['case_answer'],
      chat_history = chat_history
  ))
    ----------------------

  print('\n')

  # examiner_response = hfapi_decode(simulator_prompt, model_name=examiner_model, max_length=max_length)
  examiner_response = chatgpt_decode(simulator_prompt, model_name=examiner_model, max_length=max_length)

  response_history.append(examiner_response)
  assert 'Response to student:' in examiner_response, "Examiner response must include 'Response to student:'"
  examiner_response = examiner_response.split('Response to student:')[-1].strip()
  chat_history += f"\nExaminer: {examiner_response}"
  print(f"\nExaminer: {examiner_response}", end='')

  if 'examination finished' in chat_history.lower():
    break
      
  i += 1
  print('\n\n===')

Examiner: A 65-year-old man comes to the clinic complaining of left leg pain for three months.

Student: Could you please provide more details about the nature of the pain? Is it constant or intermittent? Does it radiate anywhere? What is the severity of the pain on a scale from 0 to 10? Are there any aggravating or relieving factors, including changes with position or activity?


Examiner: The pain is characterized by a continuous cramping in the left calf that starts after walking two blocks and goes away with rest.

===


Student: Does the patient have a history of smoking, diabetes, hypertension, or hyperlipidemia? Can you describe the findings of the peripheral vascular examination, including the presence of femoral or distal pulses, any bruits, and skin changes or temperature differences between the legs?


Examiner: The patient has a history of hypertension and type II diabetes mellitus. He quit smoking 5 years ago. The physical exam shows symmetric legs without swelling, rednes

In [9]:
for r in response_history:
    print(r)
    print("============")

["Reasoning:\nThe patient's complaint of left leg pain for three months is a chronic issue that needs further characterization. Key information to gather includes the nature, location, intensity, and radiation of the pain; exacerbating and relieving factors; associated symptoms such as swelling, redness, or warmth; history of trauma; and any previous treatments or investigations. This information will help narrow down the differential diagnoses, which could include musculoskeletal conditions (e.g., arthritis, tendinitis), vascular issues (e.g., peripheral arterial disease, deep vein thrombosis), neurological conditions (e.g., sciatica, peripheral neuropathy), or referred pain from the spine or hip.\n\nResponse to examiner:\nCould you please describe the nature of your leg pain (e.g., sharp, dull, aching)?"]
Reasoning:
The patient's complaint of left leg pain for three months is a chronic issue that requires a detailed history to understand the characteristics of the pain, any exacerbat

In [10]:
  agent_prompt = str_to_msgs(agent_prompt_template.format(
      case_prompt = test_case['case_prompt'],
      chat_history = chat_history
  ))


In [15]:
print(agent_prompt[1]['content'])

Your job is to examine the patient, who has presented with the INITIAL CASE below. You will do this by exchanging messages with an "Examiner", who is evaluating your performance. Your investigation so far is detailed in the INVESTIGATION HISTORY below. 

You are to continue your investigation and management until either you or the examiner determine that the examination is finished. You are required to give short, succinct responses to the examiner (no more than two lines of text), but your reasoning may be as long as you'd like. Your answer should be formatted as follows:

    Reasoning: [Your reasoning, including pros/cons of relevant alternatives]
    Response to examiner: [Your response to the examiner]

In your response to examiner, the items you want to investigate and your management plan needs to be as specific as possible. For any information in history, examination, or investigation that will be of use to you, you should phrase it as a question.

Here is the initial case and 